# Instruction Tuning
### Advanced Fine-Tuning Paradigms  ·  Colab T4 (16 GB) ready

> **CPT** (previous notebook) moves the model's prior `p(x)` with an **un-masked** loss over raw text — it installs *knowledge*.
> **Instruction Tuning** runs the *same* token-level cross-entropy, but **masked to the completion**, over a **broad mixture of many different tasks phrased as instructions** — it installs the *`instruction → behaviour` mapping*.
> The loss function is not the innovation here. **The mixture is the algorithm.** Zero-shot generalisation is an emergent property of *task diversity*, and this notebook measures it rather than asserting it.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)

- **Instruction tuning (IT)** — the supervised stage introduced as **FLAN** (*Wei et al., 2022, "Finetuned Language Models Are Zero-Shot Learners"*, 62 tasks) and **T0** (*Sanh et al., 2022, "Multitask Prompted Training Enables Zero-Shot Task Generalization"*, the P3 prompt collection), scaled in *Chung et al., 2022, "Scaling Instruction-Finetuned Language Models"* (**1,836 tasks**) and *Wang et al., 2022, "Super-NaturalInstructions"* (**1,616 tasks**).
- Formally, minimise token-level cross-entropy over a **mixture `M` of `T` tasks**, where each example is verbalised through an instruction **template** and **only the completion is supervised**:
  $$\mathcal{L}_{\text{IT}}(\theta) = -\,\mathbb{E}_{(I,\,x,\,y)\,\sim\,M}\Big[\sum_{t=1}^{|y|}\log \pi_\theta\big(y_t \mid \underbrace{\texttt{template}(I, x)}_{\text{masked: } -100},\; y_{<t}\big)\Big]$$
- **The objective is identical to SFT's.** What makes it *instruction tuning* rather than task-specific SFT is a property of the **dataset**, not the code: breadth of tasks, diversity of instruction phrasings, and a shared role protocol (`system`/`user`/`assistant`).
- Read as **meta-learning across task space**: instead of fitting `T` separate task functions, the model fits the single function `instruction ⟼ behaviour`. A held-out task is then **interpolation in task space** rather than extrapolation — which is precisely why zero-shot works at all.
- The five sub-mechanisms that actually decide whether an IT run works (all five appear in Section 3):
  1. **Completion-only loss masking** (`-100` on every prompt token) — the prompt is *conditioning*, not a target.
  2. **Chat-template / role protocol installation** — IT teaches a wire format that inference must then reproduce **verbatim**.
  3. **EOS supervision** — the stop token must live *inside* the supervised span, or the model never terminates.
  4. **Capped examples-proportional mixing** — cap per-task counts so a large task cannot dominate the gradient.
  5. **Held-out task-cluster evaluation** — the only honest way to substantiate the zero-shot claim (FLAN/T0 protocol).
- The counter-intuitive empirical law, and the single most useful thing to know: **generalisation to unseen tasks scales with the *number of tasks*, and saturates almost immediately in *examples per task*.** *Zhou et al., 2023 ("LIMA")* pushes this to its limit — 1,000 curated examples are enough — and formalises it as the **Superficial Alignment Hypothesis**: knowledge comes from pretraining, IT only selects the response distribution. **Spend the budget on breadth, not depth.**

### One-sentence definition of the mechanics

> **Instruction tuning fine-tunes a base model with a completion-masked causal-LM loss over a capped, diversity-maximised mixture of many distinct tasks rendered as instruction/response pairs in a fixed chat protocol, so the model learns the general mapping from instruction semantics to behaviour — and therefore performs tasks it never saw during training.**

### The exact engineering problem it solves

- **A base model has capability but no interface.** It is a text continuator: prompt it with a question and it plausibly continues *the question*, or invents more questions. The knowledge is in there — there is no *addressing scheme* for it.
- **Prompt engineering doesn't fix that**, it only works around it. Few-shot prompting burns context on every call, is brittle to formatting, and still fails on tasks whose format the model has never seen.
- **Per-task fine-tuning doesn't scale.** `N` tasks means `N` checkpoints, `N` deployments, `N` labelled datasets, and **zero** generalisation to task `N+1`. IT collapses that to **one** model that generalises.
- **It is also the mandatory prerequisite for everything downstream.** DPO/ORPO/SimPO all need a policy that can already follow an instruction — you cannot express a preference between two completions if the model cannot produce two coherent completions. IT is the stage that makes preference optimisation meaningful.
- **And the failure it prevents in the other direction:** using IT to teach *facts*. IT's dataset is 10³–10⁵ examples; that is 3–4 orders of magnitude too small to move knowledge (see the CPT notebook), so instruction pairs about facts the model does not hold teach it the **shape of a confident answer** — i.e. fluent hallucination. **CPT installs knowledge, IT installs the interface.**

---

### The Human Element — Hugging Face datasets for Instruction Tuning

| HF path | What it is | Why it's structured this way for IT |
|---|---|---|
| **`allenai/tulu-3-sft-mixture`** (939,343 rows) | The flagship **open** post-training mixture (Tülu 3): chat, math, code, precise instruction-following, safety — every row tagged with its originating `source`. | Rows are **`messages` lists**, not `(instruction, output)` pairs, because modern IT trains the **role protocol** (`system`/`user`/`assistant`), not just a Q→A mapping — multi-turn structure has to be in the data or the model never learns to hold a conversation. The **`source` column is the load-bearing design choice**: mixture composition is the only lever that reliably moves zero-shot performance, so the dataset is built to be *ablated* (drop a source, re-train, measure) rather than merely consumed. |
| **`databricks/databricks-dolly-15k`** (15,011 rows) | 100 % **human-written**, commercially licensed (CC-BY-SA), with an explicit **`category`** column over 8 task types (`open_qa` 3,742 · `general_qa` 2,191 · `classification` 2,136 · `closed_qa` 1,773 · `brainstorming` 1,766 · `information_extraction` 1,506 · `summarization` 1,188 · `creative_writing` 709). | That **`category`** column is what makes **FLAN-style held-out task-cluster evaluation** possible on a Colab budget: train on 5 clusters, evaluate on 3 the model has provably never seen. The 5.3× skew between the largest and smallest cluster is also a real, in-miniature version of the problem that **capped mixing** solves. And `context` is a **separate field from `instruction`** on purpose — IT must learn to distinguish *the task* from *the material the task operates on*. |
| **`HuggingFaceH4/no_robots`** (10k) | Human-written (**no** teacher distillation), 10 categories (`Generation`, `Rewrite`, `Summarize`, `Classify`, `Extract`, `Coding`, `Chat`, `Brainstorm`, …), `messages` format. | Deliberately **small and maximally diverse** — the practical embodiment of the LIMA claim. It exists to prove that instruction *following* is cheap (~10⁴ examples) provided the examples are high-quality and spread across many task types. Being human-written, it is also free of the teacher-model stylistic tics that distilled sets bake in. |
| **`SetFit/ag_news`** (**eval only**, 4-way topic classification) | `text` / `label_text` ∈ {World, Sports, Business, Sci/Tech}. | The **held-out benchmark**, never mixed into training. Its fixed, small label set is what allows **rank classification** — score each candidate label's log-likelihood and take the argmax — which is *exactly* the zero-shot protocol FLAN and T0 report, and it needs no generation, no judge model, and no ROUGE. This is how you get an objective number instead of a vibe. |
| **`tatsu-lab/alpaca`** · **`Muennighoff/flan`** *(classics)* | 52k self-instruct distillation from `text-davinci-003`; the original ~1,800-task FLAN v2 academic mixture. | Alpaca is the format that made IT cheap and is still the reference `(instruction, input, output)` schema; FLAN v2 is the reference for what "massive task diversity" actually looks like when you templatise academic NLP datasets 10 ways each. |

**Why they all look the same at the interface (`messages` or `instruction`/`response`) and yet nothing else about them matters more than *how you blend them*:** the loss cannot see task labels, sources, or categories — it sees tokens. Every bit of IT's engineering leverage therefore lives in **which** examples you show, in **what proportion**, and in **how many distinct phrasings** — i.e. mixture design, not model code.

> This notebook trains on a **capped 3-source mixture** (`databricks/databricks-dolly-15k` + `HuggingFaceH4/no_robots` + a streamed slice of `allenai/tulu-3-sft-mixture`), from the **`Qwen/Qwen2.5-0.5B` base** checkpoint, and measures zero-shot generalisation on **held-out dolly task clusters** plus **`SetFit/ag_news`**.

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the engineering and mathematical reason for this implementation

- **Why diversity, not volume, is the hyperparameter.** Both FLAN and Super-NaturalInstructions report the same shape: held-out-task performance rises ~log-linearly with the **number of training tasks** and **plateaus fast** in examples-per-task (FLAN-T5 captures most of its gain by ~282 tasks; SNI saturates after a few hundred instances per task). The production corollary is concrete and slightly uncomfortable: **capping a task at K examples and spending the remainder on new tasks strictly dominates** adding more rows of what you already have.
- **Why the mixture must be *capped*, mathematically.** Under raw examples-proportional mixing, task `i` contributes weight `n_i / Σ n_j`, so the gradient is dominated by whichever task happens to be biggest — dolly's `open_qa` alone is **5.3×** `creative_writing`. Uniform mixing over-corrects and over-weights tiny, noisy tasks. The T0/FLAN answer is **examples-proportional with a cap `K`**: `w_i = min(n_i, K) / Σ_j min(n_j, K)` — proportional in the tail, flat in the head.
- **Why the loss is masked to the completion (`-100`).** The prompt is *given* at inference; you never generate it. Supervising it (a) spends capacity modelling instruction text, (b) actively rewards **memorising templates** — the opposite of the generalisation you are buying, and (c) dilutes the gradient with tokens whose conditional distribution is irrelevant.
  - Honest caveat: this is a strong default, not a law. *Huerta-Enochian & Ko, 2024* find non-zero **prompt-loss weighting** can help when completions are very short relative to prompts. Mask by default; measure if your data is prompt-heavy.
- **Why `EOS` supervision is not a detail.** The model learns *when to stop* only if the stop token falls inside the supervised span. This is the single most common IT bug, and **this exact base checkpoint carries the trap**: `Qwen2.5-0.5B`'s `eos_token` is `<|endoftext|>` (151643) while its ChatML template terminates every turn with **`<|im_end|>` (151645)**. Train on the template, generate with the config's EOS, and the model **never stops** — it produces a turn and then hallucinates the next `user` turn forever. Step 1 fixes this explicitly.
- **Why the chat template is a *protocol*, not formatting sugar.** IT installs a wire format. At inference you must reproduce it **verbatim** — same special tokens, same `add_generation_prompt` behaviour. A template mismatch between training and serving is the most common cause of "my fine-tune scored worse than the base model".
- **Why over-length examples are *dropped* rather than truncated.** A truncated completion is a training example whose target ends **without** `EOS`, i.e. an explicit demonstration of *not stopping*. A handful of those is enough to produce a model that rambles. Dropping is both lazier and correct.
- **Why the base model, not the Instruct model.** `-Instruct` checkpoints have already been instruction-tuned *and* preference-optimised; re-running IT on top mostly overwrites a better job with a smaller mixture. IT belongs directly after pretraining/CPT — ideally on top of the adapter from the CPT notebook.

#### VRAM & Compute Impact

- **Per-step memory is the same as any SFT run** at the same sequence length — IT is not architecturally expensive. Its distinctive cost is **token efficiency**, and it is *worse* than CPT's on both axes:

  | | CPT (previous notebook) | **Instruction Tuning (here)** |
  |---|---|---|
  | Loss mask | none — `labels = input_ids` | **`-100` on all prompt tokens** |
  | Supervised ÷ forwarded tokens | **~100 %** | **~25–40 %** (measured in Step 3) |
  | Padding | none (packed) | **real** — variable lengths, `pad_to_multiple_of=8` (measured in Step 3) |
  | Dataset scale | 10⁹–10¹¹ tokens | **10⁶–10⁸ tokens** |
  | LoRA rank needed | **r = 64+** (knowledge is high-rank) | **r = 16** (behaviour is low-rank) |
  | Wall clock | days–weeks | **minutes–hours** |

  The two rows in the middle are why **Data Packing Strategies** — the next concept in this chapter — exists: at IT's sequence-length variance, padding plus masking can mean you pay for 3–4 tokens per token of actual supervision.
- **`r=16` is not a compromise here, it is the right size.** Per the Superficial Alignment Hypothesis, IT is selecting a response distribution the model can already express; that update is genuinely low-rank. Contrast with CPT, where low rank is a real limitation (*Biderman et al., 2024*).
- **Concrete T4 budget for this notebook** (`Qwen2.5-0.5B`, 4-bit NF4, `max_length=768`, batch 2 × accum 8):

  | Component | Cost |
  |---|---|
  | Base weights (4-bit NF4 + double quant) | ~0.40 GB |
  | LoRA r=16, all 7 projections × 24 layers | ~8.8 M params → ~0.10 GB with `paged_adamw_8bit` |
  | Activations @ 768 × 2, gradient-checkpointed | ~1.0–1.5 GB |
  | **Peak** | **~2.5–3.5 GB** — the T4 is *not* the constraint for IT |
  | ~3k examples, 1 epoch | **~10–20 min** |

- **Sequence-length policy matters more than batch size.** Instruction data is bimodal: short chat turns next to long `information_extraction` contexts. `max_length` is therefore a *data* decision — too low silently deletes your long-context tasks (the printout in Step 3 reports exactly how many).
- **Evaluation is a real compute line item.** Rank classification over `C` label options costs `C` forward passes per example (4 × 100 × 2 templates = 800 forwards here, ~1 min). Generative eval is far more expensive and needs a judge; rank classification is the cheap, objective option whenever the task has a closed label set.

#### Pros & Cons

**Pros**
- **The only stage that produces an *interface*.** One model, any task, specified in natural language — instead of `N` fine-tunes for `N` tasks.
- **Cheap and fast.** 10³–10⁴ good examples and minutes-to-hours of compute, because it teaches format rather than knowledge.
- **Zero-shot generalisation to genuinely unseen tasks** — the measurable claim, and this notebook measures it.
- **Composable and auditable.** Mixtures are just concatenations; sources can be added, dropped, capped and ablated independently (which is exactly why Tülu 3 ships a `source` column).
- **Unlocks the whole downstream stack** — DPO/ORPO/SimPO/RLHF all presuppose an instruction-following policy.
- **Standardises deployment** — one chat protocol for every task.

**Cons**
- **It cannot add knowledge — and will fluently fake it.** Instruction pairs over facts the model does not hold produce confident hallucination. Needs CPT (or retrieval) underneath.
- **Mixture engineering *is* the work, and attribution is hard.** Which of 40 sources bought which point of held-out accuracy? Answering that properly means many training runs.
- **Held-out claims are easily contaminated.** With a 939k-row opaque mixture like Tülu 3, you cannot verify that your "unseen" task is unseen. Rigorous cluster hold-out requires *per-source, per-category* exclusion — and this notebook shows how easy the leak is to miss (`no_robots/Summarize` would have leaked the held-out `summarization` cluster).
- **Catastrophic forgetting if the mixture is narrow** — heavy tuning on one task type degrades general conversational ability. The mitigation is a blend, which is the *next* concept: **MTFT**.
- **Distilled data inherits the teacher's ceiling** *and* its licence terms; style transfers along with capability (the "as an AI language model" tic is an artefact of exactly this).
- **Length inflation.** IT nudges models toward longer, more padded answers because verbose responses are over-represented in the data — watch response-length distributions, not just loss.
- **Format lock-in.** The trained template becomes an API contract; changing it later is a breaking change.

#### Metrics to watch (there is no reward signal in IT either)

- **`eval_loss` on held-out task clusters** — not on a random split of the training mixture. A random split measures memorisation of the mixture; a **cluster** hold-out measures the thing you actually claim.
- **Zero-shot rank-classification accuracy** vs. **two** references: random chance (25 % for AG News) and the **untuned base model**. The base-model delta is the only number that isolates IT's contribution.
- **Prompt-format robustness** — same task, two different instruction phrasings. A large gap means the model memorised a template instead of learning to follow instructions.
- **EOS termination rate** — fraction of generations that stop on their own inside the budget. Base ≈ 0 %, correctly tuned ≈ 100 %. The fastest smoke test that IT worked at all.
- **Supervised-token fraction** and **padding fraction** — your effective cost per unit of supervision.
- **Response-length distribution** — for detecting the length inflation above.

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**QLoRA (4-bit NF4) + capped 3-source instruction mixture + ChatML protocol + explicit `-100` completion masking, driven by `transformers.Trainer`.**

> ⚙️ **Why explicit masking instead of `SFTTrainer`'s flags:** TRL can do this for you (`completion_only_loss` / `assistant_only_loss`), but the masking boundary, the EOS placement and the drop-vs-truncate policy **are** the algorithm here — and the `<|im_end|>` vs `<|endoftext|>` trap below is invisible if a helper handles it. Step 3 builds the labels by hand and then **decodes the mask back out to prove it is correct**. Same reasoning as the CPT notebook: the data path is the concept.

**Executable pipeline:**

| Step | What | IT-specific detail |
|---|---|---|
| 1 | 4-bit **`Qwen/Qwen2.5-0.5B`** (base) + **ChatML/EOS repair** | `eos_token` ≠ turn terminator — fix it or nothing ever stops |
| 2 | LoRA **r=16** | behaviour is low-rank (contrast: CPT needed r=64) |
| 3 | **Capped** mixture: dolly (5 clusters) + no_robots + Tülu 3 slice | `min(n_i, K)` mixing; **3 clusters held out of every source** |
| 4 | Chat template + **`-100` prompt masking**, over-length **dropped** | then decoded back out and verified |
| 5 | `Trainer`, 1 epoch, cosine, `DataCollatorForSeq2Seq` | dynamic padding to multiples of 8 |
| 6 | **Zero-shot eval**: rank classification on `SetFit/ag_news`, **2 prompt templates**, base vs tuned | the FLAN/T0 protocol |
| 7 | Held-out-cluster loss · EOS termination rate · export · chat inference | it's an assistant now |

### Environment Setup

In [ ]:
%pip install torchao==0.16.0 transformers datasets peft accelerate bitsandbytes

In [ ]:
import os, gc, math, time
from collections import Counter
from itertools import islice

import torch
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(42)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# ---- Hardware-aware dtype selection (do NOT hardcode bf16 on a T4) ----------------
# The Colab T4 is Turing (compute capability 7.5). bfloat16 TENSOR CORES only exist on
# Ampere (SM 8.0) and newer; forcing bf16 makes the bitsandbytes 4-bit dequant path return
# garbage, which shows up as NaN loss or a run that trains to nothing.
major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"device capability sm_{major}{_minor} | bf16 usable: {USE_BF16} | compute dtype: {COMPUTE_DTYPE}")

# ---- The knobs that define the MIXTURE (the actual algorithm of instruction tuning) ----
MAX_LENGTH = 768   # prompt+completion budget. Instruction data is bimodal; this is a DATA decision.
PER_TASK_CAP = 200   # examples-proportional-with-cap: w_i = min(n_i, K) / sum_j min(n_j, K)
TULU_TAKE = 500   # streamed slice of the 939k-row Tulu 3 mixture
N_ZEROSHOT = 100   # held-out AG News examples for rank classification (x4 labels x2 templates)

# FLAN/T0-style held-out task CLUSTERS — excluded from EVERY source, not just the obvious one.
HELDOUT_DOLLY = {"summarization", "information_extraction", "classification"}
# The same three clusters under no_robots' own naming scheme. Missing this is THE classic
# contamination leak: "summarization" is held out from dolly but walks back in via no_robots.
HELDOUT_NO_ROBOTS = {"Summarize", "Extract", "Classify"}

### Step 1 — Base checkpoint, 4-bit quantization, and the **ChatML / EOS repair**

Two decisions here, and the second one is the bug that silently ruins most first attempts at instruction tuning.

- **`Qwen/Qwen2.5-0.5B`, not `-Instruct`.** IT is the stage that *creates* an Instruct model. If you have run the CPT notebook, the ideal base is that domain-adapted checkpoint — IT after CPT, never instead of it.
- **The termination trap.** Qwen ships the ChatML `chat_template` even on base checkpoints, so `apply_chat_template` works — but the base checkpoint's `eos_token` is **`<|endoftext|>` (151643)** while the template ends every turn with **`<|im_end|>` (151645)**. Consequences if you don't repair it:
  - Training: the token the model must learn to emit to stop is `<|im_end|>`, which is fine — it comes from the template.
  - **Generation: `generate()` watches `eos_token_id` = `<|endoftext|>`, which the model now never emits → it produces its answer and then keeps going, hallucinating the next `user` turn until it hits `max_new_tokens`.**
  - The fix is one line: point `eos_token` at the actual turn terminator, and keep `pad_token` distinct so the collator's padding can never be confused with a real stop.

In [ ]:
base_model_id = "Qwen/Qwen2.5-0.5B"   # BASE. Instruction tuning is what MAKES an -Instruct model.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,   # fp16 on T4 (Turing has no bf16 tensor cores)
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# ---- Install / verify the chat protocol -----------------------------------------
CHATML = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)
if tokenizer.chat_template is None:
    # Some base checkpoints ship no template at all — then YOU choose the protocol, and this
    # is the moment it becomes an API contract for every future inference call.
    tokenizer.chat_template = CHATML
    print("no chat_template found -> installed minimal ChatML")
else:
    print("chat_template present on the base checkpoint (Qwen ships ChatML)")

# ---- THE TERMINATION TRAP -------------------------------------------------------
TURN_END = "<|im_end|>"
turn_end_id = tokenizer.convert_tokens_to_ids(TURN_END)
print(f"template terminates turns with {TURN_END!r} (id {turn_end_id})")
print(f"tokenizer.eos_token is {tokenizer.eos_token!r} (id {tokenizer.eos_token_id})")

if tokenizer.eos_token != TURN_END:
    # Without this, generate() waits for <|endoftext|> — a token the tuned model never emits —
    # and every response runs to max_new_tokens, hallucinating the next user turn.
    print(f"  MISMATCH -> repointing eos_token to {TURN_END!r} so generate() actually stops")
    tokenizer.eos_token = TURN_END
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    # Keep PAD distinct from EOS: the collator masks pads to -100, and a shared id would make
    # "stop here" and "ignore this" the same token.
    tokenizer.pad_token = "<|endoftext|>"
print(f"final: eos={tokenizer.eos_token!r}({tokenizer.eos_token_id}) pad={tokenizer.pad_token!r}({tokenizer.pad_token_id})")

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",   # memory-efficient attention; FlashAttention-2 needs Ampere+
)
model.config.use_cache = False  # required with gradient checkpointing
model.config.eos_token_id = tokenizer.eos_token_id   # keep the model config in sync with the tokenizer
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"\n{model.get_memory_footprint()/1e9:.2f} GB base (4-bit)")

### Step 2 — LoRA sized **for behaviour**, not for knowledge

Deliberately **`r=16` — one quarter of the rank the CPT notebook used.** This is not cost-cutting:

- Per the **Superficial Alignment Hypothesis**, IT selects a response distribution the model can *already express*. That update is genuinely **low-rank**; adding rank mostly adds overfitting surface and optimiser state.
- Still **all seven projections** — instruction following is distributed across attention *and* MLP, and the marginal cost at `r=16` is trivial.
- **`lora_dropout=0.05`** matters more here than in CPT: IT datasets are small (10³–10⁴ examples), so template memorisation is a live risk.

In [ ]:
peft_config = LoraConfig(
    r=16,  # behaviour is LOW-RANK (CPT needed r=64 because knowledge is not)
    lora_alpha=32,  # alpha = 2r
    lora_dropout=0.05,  # small dataset -> real overfitting/template-memorisation risk
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # reentrant=True breaks PEFT + checkpointing
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### Step 3 — Build the **mixture**: 3 sources, capped, with 3 task clusters held out of *every* source

This is the actual algorithm of instruction tuning. Three things happen here, and the third is the one people get wrong:

1. **Normalise every source to one shape** — `{"messages": [...], "source": ..., "task": ...}`. The loss cannot see `source`/`task`; they exist purely so the *mixture* is auditable (exactly why Tülu 3 ships a `source` column).
2. **Cap each task at `PER_TASK_CAP`** — `w_i = min(n_i, K) / Σⱼ min(n_j, K)`. Proportional in the tail, flat in the head. Without it, dolly's `open_qa` (3,742) outweighs `creative_writing` (709) by **5.3×** and the gradient follows the biggest task rather than the broadest signal.
3. **Exclude the held-out clusters from *every* source, under each source's own naming scheme.** `summarization` is held out of dolly — but `no_robots` calls the same task `Summarize`, and forgetting that leaks the eval straight back into training. This is the most common way a "zero-shot" number turns out to be meaningless.
   - ⚠️ **Honest limit:** Tülu 3 is a 939k-row blend of dozens of sources; it *certainly* contains summarisation-like data and cannot be audited at this granularity. So the controlled hold-out here is the **dolly + no_robots** part, and `SetFit/ag_news` (Step 6) is held out by construction — it is never loaded as training data at all. Reporting **base vs. tuned on identical prompts** is what makes the measurement meaningful regardless of mixture opacity.

In [ ]:
# ---- Source 1: dolly-15k — human-written, 8 explicit task clusters ---------------
dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
print("dolly cluster sizes:", dict(Counter(dolly["category"]).most_common()))
print(f"skew (largest / smallest): {max(Counter(dolly['category']).values()) / min(Counter(dolly['category']).values()):.1f}x"
      "  <- exactly what the cap exists to flatten")

def norm_msgs(msgs):
    """Rebuild every message dict with the SAME key order.

    concatenate_datasets() compares Arrow schemas exactly, and list<struct<...>> field order
    is part of the schema — dolly's hand-built {role, content} would not merge with
    no_robots'/Tulu's stored {content, role} without this.
    """
    return [{"content": m["content"], "role": m["role"]} for m in msgs]

def dolly_to_messages(ex):
    # `context` is a SEPARATE field from `instruction` on purpose: IT has to learn the
    # difference between the task and the material the task operates on.
    user = ex["instruction"] if not ex["context"] else f"{ex['instruction']}\n\n{ex['context']}"
    return {
        "messages": norm_msgs([{"role": "user", "content": user},
                               {"role": "assistant", "content": ex["response"]}]),
        "source": "dolly",
        "task": ex["category"],
    }

dolly_norm = dolly.shuffle(seed=42).map(dolly_to_messages, remove_columns=dolly.column_names)

# Split BEFORE capping: the held-out clusters become the eval set and never touch training.
dolly_train_pool = dolly_norm.filter(lambda ex: ex["task"] not in HELDOUT_DOLLY)
dolly_heldout = dolly_norm.filter(lambda ex: ex["task"] in HELDOUT_DOLLY).select(range(60))
print(f"\ndolly train pool: {len(dolly_train_pool)} | held-out clusters {sorted(HELDOUT_DOLLY)}: {len(dolly_heldout)} eval rows")

In [ ]:
def cap_per_task(ds, cap):
    """Examples-proportional mixing WITH A CAP: keep at most `cap` rows per task.

    w_i = min(n_i, K) / sum_j min(n_j, K)  -- proportional in the tail, flat in the head.
    Raw proportional mixing lets the biggest task own the gradient; uniform mixing
    over-weights tiny noisy tasks. This is the T0/FLAN compromise.
    """
    seen, keep = Counter(), []
    for i, task in enumerate(ds["task"]):  # ds is pre-shuffled, so this is an unbiased sample
        if seen[task] < cap:
            seen[task] += 1
            keep.append(i)
    return ds.select(keep), seen

dolly_capped, dolly_counts = cap_per_task(dolly_train_pool, PER_TASK_CAP)
print(f"dolly after cap({PER_TASK_CAP}): {len(dolly_capped)} rows")
print(" ", dict(dolly_counts.most_common()))

In [ ]:
# ---- Source 2: no_robots — human-written, 10 categories, ALREADY in messages form ----
no_robots = load_dataset("HuggingFaceH4/no_robots", split="train")
print("no_robots categories:", dict(Counter(no_robots["category"]).most_common()))

def no_robots_norm(ex):
    return {"messages": norm_msgs(ex["messages"]), "source": "no_robots", "task": ex["category"]}

nr = no_robots.shuffle(seed=42).map(no_robots_norm, remove_columns=no_robots.column_names)

# THE LEAK: no_robots names the held-out clusters differently. Excluding only the dolly
# spellings would quietly reintroduce summarization/extraction/classification into training.
before = len(nr)
nr = nr.filter(lambda ex: ex["task"] not in HELDOUT_NO_ROBOTS)
print(f"\ndropped {before - len(nr)} rows in {sorted(HELDOUT_NO_ROBOTS)} "
      f"-> the same clusters held out of dolly, under no_robots' naming")

# Keep only rows that end on an assistant turn (a few are prompt-only/odd).
nr = nr.filter(lambda ex: len(ex["messages"]) >= 2 and ex["messages"][-1]["role"] == "assistant")
nr_capped, nr_counts = cap_per_task(nr, PER_TASK_CAP)
print(f"no_robots after cap({PER_TASK_CAP}): {len(nr_capped)} rows")
print(" ", dict(nr_counts.most_common()))

In [ ]:
# ---- Source 3: Tulu 3 SFT mixture — the flagship open blend, STREAMED -------------
# 939,343 rows / 1.4 GB: streaming + islice touches only the shards we actually read.
tulu_stream = load_dataset("allenai/tulu-3-sft-mixture", split="train", streaming=True)

tulu_rows = []
for row in islice(tulu_stream, TULU_TAKE * 3):  # over-fetch, then filter down
    msgs = row["messages"]
    # Single-turn only, purely to keep sequence lengths inside MAX_LENGTH on a T4.
    if len(msgs) == 2 and msgs[0]["role"] == "user" and msgs[1]["role"] == "assistant":
        tulu_rows.append({"messages": norm_msgs(msgs), "source": "tulu3", "task": row["source"]})
    if len(tulu_rows) >= TULU_TAKE:
        break

tulu = Dataset.from_list(tulu_rows)
print(f"tulu3 single-turn rows: {len(tulu)}")
print(" top sub-sources:", dict(Counter(tulu["task"]).most_common(8)))

In [ ]:
# ---- The mixture ------------------------------------------------------------------
mixture = concatenate_datasets([dolly_capped, nr_capped, tulu]).shuffle(seed=42)

print(f"MIXTURE: {len(mixture)} examples")
print("  by source:", dict(Counter(mixture["source"]).most_common()))
print(f"  distinct tasks: {len(set(mixture['task']))}   <- the number that actually drives zero-shot")
print(f"  largest single task share: {100*max(Counter(mixture['task']).values())/len(mixture):.1f}%")

# Sanity: no held-out cluster survived, under EITHER naming scheme.
leaked = HELDOUT_DOLLY.union(HELDOUT_NO_ROBOTS).intersection(set(mixture["task"]))
assert not leaked, f"held-out clusters leaked into training: {leaked}"
print("  held-out cluster leak check: clean")

### Step 4 — Chat template + **explicit `-100` completion masking**

The defining implementation detail of instruction tuning, and the exact inverse of the CPT notebook (`labels = input_ids`, nothing masked):

- **Render to text first, then tokenize the two halves separately.** `prompt_text` = template with **`add_generation_prompt=True`** (ends `<|im_start|>assistant\n`); `completion_text` = the rest of the full render — the assistant turn **plus its `<|im_end|>`**.
- **Why not tokenize once and slice at `len(prompt_ids)`?** Because token-level prefix equality does **not** hold. Qwen's pre-tokenizer merges runs of newlines (`\s*[\r\n]+`), so a response beginning with `\n` **fuses with the template's own trailing newline into a single token** — the prompt stops being a token-prefix of the full sequence and every label past the boundary silently shifts by one. Splitting on the **string** boundary makes the mask exact by construction. (Same approach TRL and Axolotl take for prompt-completion data; the only cost is that the seam tokenises as `\n` + `\n` rather than one `\n\n`.)
- **`labels = [-100] * len(prompt_ids) + completion_ids`** — prompt tokens are conditioning, only the completion carries gradient, and **the stop token sits inside the supervised span**. That is how the model learns to terminate.
- The template must still be **prefix-consistent at the string level** — asserted, not assumed, with the divergence point reported.
- **Over-length examples are dropped, not truncated.** A truncated completion is a supervised example that ends *without* `<|im_end|>` — i.e. an explicit lesson in never stopping.

In [ ]:
def build_masked_example(messages):
    """Render one conversation to input_ids + completion-only labels. Returns None if too long."""
    # ponytail: supervises only the LAST assistant turn. For multi-turn data every assistant
    # turn should carry loss, which needs a per-turn mask (a {% generation %}-aware template /
    # TRL's assistant_only_loss) rather than one prompt/completion split. Upgrade there if
    # multi-turn conversation quality matters.
    prompt_msgs, answer = messages[:-1], messages[-1]

    # Render to TEXT first, then tokenize the two halves SEPARATELY.
    # Tokenizing the full conversation once and slicing at len(prompt_ids) is NOT safe: Qwen's
    # pre-tokenizer merges runs of newlines (`\s*[\r\n]+`), so a response that starts with "\n"
    # FUSES with the template's own trailing newline into a single token. The prompt then stops
    # being a token-level prefix of the full sequence and every label past the boundary shifts
    # by one. Splitting on the STRING boundary makes the mask exact by construction.
    prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(prompt_msgs + [answer], tokenize=False)
    if not full_text.startswith(prompt_text):
        # String-level divergence IS a real template bug (unlike the token-level case above).
        c = len(os.path.commonprefix([prompt_text, full_text]))
        raise ValueError(
            f"chat template is not prefix-consistent at char {c}: "
            f"prompt has {prompt_text[c:c+40]!r}, full has {full_text[c:c+40]!r}"
        )
    completion_text = full_text[len(prompt_text):]   # the assistant turn + its <|im_end|>

    # add_special_tokens=False: no automatic BOS/EOS. The <|im_*|> markers are ADDED tokens,
    # so they still tokenize to single ids regardless of this flag.
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(completion_text, add_special_tokens=False)["input_ids"]

    if len(prompt_ids) + len(completion_ids) > MAX_LENGTH:
        return None            # DROP, never truncate: a cut completion teaches "don't stop"

    return {
        "input_ids": prompt_ids + completion_ids,
        "attention_mask": [1] * (len(prompt_ids) + len(completion_ids)),
        # -100 is CrossEntropyLoss's ignore_index: prompt tokens are CONDITIONING, not targets.
        # completion_ids keeps its trailing <|im_end|>, so the STOP TOKEN IS SUPERVISED.
        "labels": [-100] * len(prompt_ids) + completion_ids,
    }

def tokenize_mixture(ds, desc):
    rows, dropped = [], 0
    for msgs in ds["messages"]:
        ex = build_masked_example(msgs)
        if ex is None:
            dropped += 1
        else:
            rows.append(ex)
    print(f"{desc}: kept {len(rows)}, dropped {dropped} over {MAX_LENGTH} tokens "
          f"({100*dropped/max(1, len(rows)+dropped):.1f}% — these are your long-context tasks)")
    return Dataset.from_list(rows)

train_tok   = tokenize_mixture(mixture, "train mixture")
heldout_tok = tokenize_mixture(dolly_heldout, "held-out clusters")
assert len(train_tok), "every training example exceeded MAX_LENGTH — raise MAX_LENGTH"
assert len(heldout_tok), f"all held-out rows exceeded MAX_LENGTH={MAX_LENGTH} — raise it or hold out more rows"

In [ ]:
# ---- PROVE the mask is right, by decoding it back out ----------------------------
# A silently-wrong mask boundary trains fine and produces a subtly broken model. Look at it.
ex = train_tok[0]
sup = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
masked = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l == -100]

print("=== MASKED (conditioning, label -100) ===")
print(repr(tokenizer.decode(masked)))
print("\n=== SUPERVISED (label != -100) ===")
print(repr(tokenizer.decode(sup)))
print(f"\nends on the turn terminator: {tokenizer.decode(sup[-1:])!r} == {TURN_END!r} -> "
      f"{tokenizer.decode(sup[-1:]) == TURN_END}  <- this is what teaches termination")

# Label alignment: HF shifts internally, so labels[i] must equal input_ids[i] wherever supervised.
assert all(l == t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100), "label/token misalignment"

# ---- What all this costs: supervised-token fraction + padding waste --------------
tot = sum(len(r) for r in train_tok["input_ids"])
sup_tot = sum(sum(1 for l in r if l != -100) for r in train_tok["labels"])
lens = [len(r) for r in train_tok["input_ids"]]

# Padding cost of dynamic padding at batch size 2. This is the WORST case: `group_by_length=True`
# in Step 5 batches similar lengths together, which cuts it further (measure both if you care).
bs = 2
padded = sum(max(lens[i:i + bs]) * len(lens[i:i + bs]) for i in range(0, len(lens), bs))

print(f"\nexamples: {len(train_tok)}")
print(f"median / max length: {sorted(lens)[len(lens)//2]} / {max(lens)} (budget {MAX_LENGTH})")
print(f"real tokens: {tot/1e6:.2f}M")
print(f"SUPERVISED tokens: {sup_tot/1e6:.2f}M -> {100*sup_tot/tot:.1f}% of real tokens carry gradient")
print(f"padding (dyn, bs={bs}): {100*(padded-tot)/padded:.1f}% of forwarded tokens are PAD "
      f"(worst case — group_by_length reduces this)")
print(f"=> supervision per forwarded token: {100*sup_tot/padded:.1f}%  (CPT's packed+unmasked run: ~100%)")
print("^ this ratio is the entire motivation for the next concept: Data Packing Strategies.")

### Step 5 — `TrainingArguments`, dynamic-padding collator, `Trainer`

- **`DataCollatorForSeq2Seq`** is doing exactly the right job for a decoder-only IT run, so there is no hand-written collator here: it pads `input_ids` with `pad_token_id`, `labels` with **`-100`**, builds the `attention_mask`, and rounds to `pad_to_multiple_of=8` for tensor-core alignment.
- **`num_train_epochs=2`** — unlike CPT's strict single pass. IT datasets are small; 2–3 epochs is the standard range (FLAN/Tülu recipes), and eval loss on the held-out clusters is what tells you when to stop.
- **`learning_rate=2e-4`, cosine, `warmup_steps` ≈ 3 %** — LoRA-scale. `warmup_steps` is a step count, not a ratio, so it is derived from the real optimizer-step total and rescales automatically when the mixture size changes. A full fine-tune of IT sits at **1e-5–2e-5**.
- **`eval_dataset=heldout_tok`** — the held-out *task clusters*, not a random split of the mixture. A random split measures memorisation of the mixture; the cluster hold-out measures the generalisation you are actually claiming.

In [ ]:
per_device_train_batch_size = 2
gradient_accumulation_steps = 8  # effective batch = 16 conversations/optimizer step
num_train_epochs = 2  # IT data is small; 2-3 epochs is the standard range

total_optimizer_steps = math.ceil(
    len(train_tok) * num_train_epochs / (per_device_train_batch_size * gradient_accumulation_steps)
)
warmup_steps = max(10, int(0.03 * total_optimizer_steps))   # ~3% warmup, derived from the real step count
eval_steps = max(20, total_optimizer_steps // 4)
print(f"examples {len(train_tok)} | optimizer steps {total_optimizer_steps} | "
      f"warmup {warmup_steps} | eval every {eval_steps}")

In [ ]:
# Pads input_ids with pad_token_id AND labels with -100 (label_pad_token_id) — the two
# different pad values that a hand-rolled collator usually gets wrong.
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,  # decoder-only: do NOT build decoder_input_ids
    padding=True,  # dynamic: pad to the longest sequence IN THE BATCH
    label_pad_token_id=-100,
    pad_to_multiple_of=8,  # tensor-core friendly shapes
)

training_args = TrainingArguments(
    output_dir="./it_output",
    run_name="instruction-tuning-t4",

    num_train_epochs=num_train_epochs,
    learning_rate=2e-4,  # LoRA scale; a full FT of IT would be 1e-5..2e-5
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,  # a step count, not a ratio — derived in the cell above
    weight_decay=0.0,  # LoRA + 2 epochs: decay buys little, costs tuning time
    max_grad_norm=1.0,

    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    group_by_length=True,  # batch similar lengths together -> less padding waste

    # ---- T4 16 GB hardening ----
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=USE_BF16, fp16=not USE_BF16,
    optim="paged_adamw_8bit",
    dataloader_num_workers=2,

    # ---- Eval on HELD-OUT TASK CLUSTERS, not a random split of the mixture ----
    eval_strategy="steps",
    eval_steps=eval_steps,
    per_device_eval_batch_size=2,

    logging_steps=10,
    save_strategy="no",
    report_to="none",
    label_names=["labels"],
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=heldout_tok,  # summarization + information_extraction + classification
    data_collator=collator,
    processing_class=tokenizer,
)

### Step 6 — The zero-shot harness: **rank classification** on `SetFit/ag_news`

This is the measurement that makes the concept's central claim falsifiable, and it is the **exact protocol FLAN and T0 report**:

- Build an instruction naming the four label options, then **score each option's log-likelihood as a continuation** and take the **argmax**. No generation, no parsing, no judge model — just `C` forward passes per example.
- **Length-normalised** (mean log-prob per option token) so that `Sci/Tech` isn't penalised for tokenising longer than `World`.
- **Two different prompt templates**, because "follows instructions" and "memorised one template" are different things and only a phrasing sweep tells them apart.
- **AG News is never loaded as training data** — this is the one hold-out no mixture opacity can compromise.
- Run **now**, before training: LoRA's `B` matrices are zero-initialised, so the adapter is an exact identity and this measures the **untouched base model**.

In [ ]:
ag_full = load_dataset("SetFit/ag_news", split="test")
LABELS = sorted(set(ag_full["label_text"]))
ag = ag_full.shuffle(seed=42).select(range(N_ZEROSHOT))
print("held-out task: 4-way topic classification | labels:", LABELS)
print("random-chance accuracy:", f"{100/len(LABELS):.0f}%")

# Two phrasings of the SAME task — FLAN-style template multiplication. The gap between them
# measures robustness to phrasing, i.e. instruction-FOLLOWING vs template memorisation.
TEMPLATES = {
    "direct": "Classify the topic of the following news article. Answer with one of: {opts}.\n\nArticle: {text}\n\nTopic:",
    "question": "{text}\n\nQuestion: which of these topics does this news article belong to — {opts}?\nAnswer:",
}

@torch.no_grad()
def rank_classify(model, text, template, labels):
    """FLAN/T0 rank classification: score every label as a continuation, take the argmax."""
    prompt = template.format(opts=", ".join(labels), text=text[:1200])
    # Identical chat protocol as training — a template mismatch here would silently tank the score.
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    scores = []
    for lab in labels:
        lab_ids = tokenizer(lab, add_special_tokens=False)["input_ids"]
        ids = torch.tensor([prompt_ids + lab_ids], device=model.device)
        logits = model(input_ids=ids).logits[0]
        # Position i predicts token i+1, so the logits for the label start one step early.
        lp = logits[len(prompt_ids) - 1:-1].float().log_softmax(-1)
        pos = torch.arange(len(lab_ids), device=lp.device)
        tok_lp = lp[pos, torch.tensor(lab_ids, device=lp.device)]
        scores.append(tok_lp.mean().item())   # length-normalised: don't punish multi-token labels
    return labels[max(range(len(labels)), key=lambda i: scores[i])]

def zeroshot_accuracy(model, ds, labels):
    model.eval()
    out = {}
    for name, tmpl in TEMPLATES.items():
        hits = sum(rank_classify(model, ex["text"], tmpl, labels) == ex["label_text"] for ex in ds)
        out[name] = 100 * hits / len(ds)
    return out

# Zero-init LoRA B => the adapter is the identity => this IS the base model.
t0 = time.time()
base_zs = zeroshot_accuracy(model, ag, LABELS)
print(f"\nBASE model zero-shot accuracy ({time.time()-t0:.0f}s): "
      + " | ".join(f"{k} {v:.1f}%" for k, v in base_zs.items()))

### Step 7 — Train

Watch **`eval_loss`** — it is computed on the **held-out task clusters** (`summarization`, `information_extraction`, `classification`), so it is a generalisation signal, not a memorisation signal. If `loss` falls while `eval_loss` climbs, the mixture is too narrow or you are past the useful epoch count.

In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()

train_result = trainer.train()

wall_min = (time.time() - t0) / 60
print(f"\nwall clock: {wall_min:.1f} min")
print(f"peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print(f"final train loss: {train_result.training_loss:.4f}")
print(f"held-out cluster loss: {trainer.evaluate()['eval_loss']:.4f}")

trainer.save_model("./instruction_tuned_adapter")
tokenizer.save_pretrained("./instruction_tuned_adapter")   # carries the repaired eos_token — REQUIRED
print("Saved -> ./instruction_tuned_adapter")

In [ ]:
# ---- Did instruction tuning actually buy zero-shot generalisation? ---------------
tuned_zs = zeroshot_accuracy(model, ag, LABELS)
chance = 100 / len(LABELS)

print(f"{'template':<12}{'chance':>9}{'base':>9}{'tuned':>9}{'delta':>9}")
print("-" * 48)
for k in TEMPLATES:
    print(f"{k:<12}{chance:>8.1f}%{base_zs[k]:>8.1f}%{tuned_zs[k]:>8.1f}%{tuned_zs[k]-base_zs[k]:>+8.1f}")

spread_base  = max(base_zs.values()) - min(base_zs.values())
spread_tuned = max(tuned_zs.values()) - min(tuned_zs.values())
print(f"\nprompt-format spread : base {spread_base:.1f}pts -> tuned {spread_tuned:.1f}pts")
print("(shrinking spread = following the instruction; growing = memorising one template)")
print(f"held-out TASK (AG News) was never in the mixture; held-out CLUSTERS "
      f"{sorted(HELDOUT_DOLLY)} were excluded from every source.")

In [ ]:
# ---- EOS termination rate: the fastest smoke test that IT worked ----------------
# A base model does not stop — it answers, then hallucinates the next user turn until the
# budget runs out. A correctly instruction-tuned model emits <|im_end|> and halts.
model.gradient_checkpointing_disable()  # training is done; checkpointing conflicts with the KV cache
model.config.use_cache = True
model.eval()

probe_prompts = [ex["messages"][0]["content"][:400] for ex in dolly_heldout.select(range(8))]
MAX_NEW = 160

@torch.no_grad()
def stopped_on_its_own(prompt, use_adapter=True):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(model.device)
    def _gen():
        out = model.generate(input_ids=ids, max_new_tokens=MAX_NEW, do_sample=False,
                             eos_token_id=tokenizer.eos_token_id,
                             pad_token_id=tokenizer.pad_token_id)
        return out.shape[-1] - ids.shape[-1] < MAX_NEW  # stopped before the budget?
    if use_adapter:
        return _gen()
    with model.disable_adapter():  # free A/B against the base weights, no 2nd model in VRAM
        return _gen()

base_stop  = sum(stopped_on_its_own(p, use_adapter=False) for p in probe_prompts)
tuned_stop = sum(stopped_on_its_own(p, use_adapter=True) for p in probe_prompts)
print(f"EOS termination within {MAX_NEW} tokens -> base {base_stop}/{len(probe_prompts)} | "
      f"instruction-tuned {tuned_stop}/{len(probe_prompts)}")

In [ ]:
# ---- Full generations, WITH special tokens visible: does <|im_end|> show up, just late? ----
@torch.no_grad()
def generate_full(prompt, use_adapter=True, max_new_tokens=MAX_NEW):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(model.device)
    def _gen():
        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
                             eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
        gen_len = out.shape[-1] - ids.shape[-1]
        # skip_special_tokens=False on PURPOSE here: we need to SEE <|im_end|> if it's in there.
        text_out = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=False)
        return text_out, gen_len < max_new_tokens, gen_len
    if use_adapter:
        return _gen()
    with model.disable_adapter():
        return _gen()

print("=== HELD-OUT cluster prompt (never trained on this task type) ===")
txt, stopped, gen_len = generate_full(probe_prompts[0], use_adapter=True)
print(f"stopped={stopped} | length={gen_len}/{MAX_NEW}")
print(txt[:600])

# ---- Same check, but on prompts from a task type that WAS in the training mixture ----
in_dist_prompts = [ex["messages"][0]["content"][:400] for ex in dolly_capped.shuffle(seed=1).select(range(8))]
in_dist_stop = sum(stopped_on_its_own(p, use_adapter=True) for p in in_dist_prompts)
print(f"\nEOS termination on IN-DISTRIBUTION prompts: {in_dist_stop}/{len(in_dist_prompts)}  "
      f"(vs. held-out clusters: {tuned_stop}/{len(probe_prompts)})")

txt2, stopped2, gen_len2 = generate_full(in_dist_prompts[0], use_adapter=True)
print(f"\nstopped={stopped2} | length={gen_len2}/{MAX_NEW}")
print(txt2[:600])

# ---- Is 160 tokens even enough for the HELD-OUT clusters' real answers? ----
gold_lens = sorted(sum(1 for l in ex["labels"] if l != -100) for ex in heldout_tok)
print(f"\ngold completion length, held-out clusters: median {gold_lens[len(gold_lens)//2]}, "
      f"max {gold_lens[-1]}  (MAX_NEW budget = {MAX_NEW})")
if gold_lens[len(gold_lens)//2] > MAX_NEW:
    print("  -> reference answers for THIS cluster routinely exceed the budget; raise MAX_NEW before concluding anything.")

---

## **[Key Observations]**

*Fill in from the runs above. For instruction tuning the headline number is always a **base vs. tuned delta on a held-out task** — a single absolute score proves nothing.*

### Mixture configuration (the actual algorithm)

| Setting | Value |
|---|---|
| Sources (rows each) | dolly ___ · no_robots ___ · tulu3 ___ |
| **Distinct tasks** in the mixture | |
| `PER_TASK_CAP` | |
| Largest single-task share (%) | |
| Held-out clusters (per source) | `summarization` / `Summarize`, `information_extraction` / `Extract`, `classification` / `Classify` |
| `MAX_LENGTH` · % dropped as over-length | |
| LoRA `r` / `alpha` / targets | |
| Epochs · LR · scheduler | |

### Zero-shot generalisation (`SetFit/ag_news`, never in the mixture)

| Template | Chance | Base | Tuned | Δ |
|---|---|---|---|---|
| `direct` | 25.0 % | | | |
| `question` | 25.0 % | | | |
| **prompt-format spread** | — | | | should **shrink** |

### Training & efficiency

| Metric | Value |
|---|---|
| Final `train_loss` | |
| **Held-out-cluster `eval_loss`** (start → end) | |
| Supervised-token fraction (%) | |
| Padding fraction at bs=2 (%) | |
| **Supervision per forwarded token (%)** | *(CPT's packed run: ~100 %)* |
| Peak VRAM (GB) · wall clock (min) | |
| **EOS termination rate** base → tuned | ___/8 → ___/8 |

### Qualitative

- Does the tuned model **answer** the instruction, where the base model continues or re-asks it?
- Does it hold format on the **held-out clusters** (summarise when asked to summarise, extract when asked to extract) despite never seeing those clusters?
- Response length: base vs tuned — is length inflation visible already at this scale?
- Any refusals/safety behaviour leaking in from the Tülu 3 slice?

### Ablations worth the compute

- **`PER_TASK_CAP` ∈ {50, 350, ∞}** at a *fixed total example count* — the direct test of "breadth beats depth".
- **Drop one source at a time** — which of the three actually buys the AG News delta?
- **`completion_only` vs. full-sequence loss** — measure, don't assume, on your prompt/completion length ratio.
- **Remove the EOS repair** in Step 1 and watch the termination rate collapse. Instructive, and it takes one line.

## Export — Download the Instruction-Tuned Adapter (Optional)

In [ ]:
import shutil, os

folder_to_zip = "./instruction_tuned_adapter"
output_filename = "instruction_tuned_adapter.zip"

shutil.make_archive(output_filename.replace(".zip", ""), "zip", folder_to_zip)
if os.path.exists(output_filename):
    print(f"File: {output_filename} | ({os.path.getsize(output_filename)/1e6:.2f} MB)")
else:
    print("Zip not found — run training + save first.")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found. Did training + zipping finish?")

---

## Model Usage — it is an **assistant** now

The CPT notebook ended with a *continuator*: give it a prefix, it extends the text. This one ends with something you can talk to — same base weights, ~3k instruction examples, one adapter.

Two things the inference path must get right, and both are consequences of Step 1:

- **The chat template is now an API contract.** Serve with the exact template used in training (`add_generation_prompt=True`), or you get a model that looks worse than the base.
- **Load the tokenizer from the adapter directory, not the hub.** It carries the **repaired `eos_token`** (`<|im_end|>`); loading the base tokenizer instead reintroduces the never-terminates bug at serving time — the classic "it worked in the notebook" failure.

The base-vs-tuned A/B is free: `PeftModel` keeps the base weights intact, so `with model.disable_adapter():` gives the untuned answer in the same process.

In [ ]:
# Free the TRAINING model before loading a second copy for inference.
import gc, torch

for _obj in ["trainer", "model"]:
    if _obj in globals():
        del globals()[_obj]
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free before inference load: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

base_model_id = "Qwen/Qwen2.5-0.5B"
adapter_path = "./instruction_tuned_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

# From the ADAPTER dir: carries the repaired eos_token AND the exact chat template we trained on.
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
model = PeftModel.from_pretrained(base, adapter_path)
model.config.use_cache = True
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.eval()
print(f"eos={tokenizer.eos_token!r}({tokenizer.eos_token_id}) | chat_template loaded: {tokenizer.chat_template is not None}")

In [ ]:
@torch.no_grad()
def chat(instruction, max_new_tokens=200, use_adapter=True):
    """Same protocol as training: chat template + add_generation_prompt=True. No exceptions."""
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}], tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(model.device)

    def _gen():
        probe = model(input_ids=ids).logits
        if not torch.isfinite(probe).all():
            raise RuntimeError("Non-finite logits — dtype/hardware mismatch (bf16 on Turing?) or a diverged run.")
        out = model.generate(
            input_ids=ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy: a fair A/B needs determinism
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,  # the REPAIRED eos — <|im_end|>
            pad_token_id=tokenizer.pad_token_id,
        )
        completion = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
        stopped = out.shape[-1] - ids.shape[-1] < max_new_tokens
        return completion.strip(), stopped

    if use_adapter:
        return _gen()
    with model.disable_adapter():
        return _gen()


# Instructions from the HELD-OUT clusters (summarization / extraction / classification) —
# task types that appear nowhere in the training mixture, under any source's naming.
held_out_instructions = [
    "Summarize the following in one sentence: The T4 is a Turing-generation GPU with 16 GB of "
    "memory and no bfloat16 tensor cores, which is why mixed-precision training on it must use "
    "float16 and why 4-bit quantization is so useful there.",
    "Extract every city name from this text as a comma-separated list: We flew from Mumbai to "
    "Berlin, spent two days in Prague, then took the train to Vienna.",
    "Is this review positive or negative? Answer with one word. Review: the battery dies in "
    "three hours and support never replied.",
]

for i, ins in enumerate(held_out_instructions, 1):
    print(f"\n[Instruction {i}] {ins[:110]}...")
    b_txt, b_stop = chat(ins, use_adapter=False)
    t_txt, t_stop = chat(ins, use_adapter=True)
    print(f"[BASE | stopped={b_stop}] {b_txt[:300]}")
    print(f"[TUNED | stopped={t_stop}] {t_txt[:300]}")
    print("-" * 70)